# Figure4_CPVAE_rank8_GO_enrichment

In [1]:

from pathlib import Path
import os, re, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import hypergeom
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset, random_split
    HAS_TORCH = True
except Exception as e:
    HAS_TORCH = False
    print('Torch unavailable:', e)

try:
    from tensorly.decomposition import parafac
    from tensorly.cp_tensor import cp_to_tensor
    HAS_TENSORLY = True
except Exception as e:
    HAS_TENSORLY = False
    print('Tensorly unavailable:', e)

project_dir = Path('/Users/sidaye/Documents/python/ST_MultiCAST')
input_dir = project_dir / 'Input'
base_output_dir = project_dir / 'Output'
model_comparison_dir = base_output_dir / 'model comparison'
ai_output_dir = base_output_dir / 'AI_spatiotemporal_models_python'
Spacepoints = ['st','SI1','SI2','SI3','SI4','SI5','SI6','SI7','SI8','SI9','ce','co']
Full_Timepoints = ['1h','3h','6h','12h','24h']
feature_order = [f'{t}_{s}' for t in Full_Timepoints for s in Spacepoints]

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'DejaVu Sans'

def save_pdf(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches='tight', transparent=True)
    plt.close(fig)

def build_data():
    df = pd.read_csv(input_dir / 'Spatial_temporal_MultiSCAST_FC_final_capping.csv')
    df['Gene'] = df['Gene'].astype(str)
    df['Time'] = df['Time'].astype(str)
    df['Space'] = df['Space'].astype(str)
    df = df[df['Time'].isin(Full_Timepoints) & df['Space'].isin(Spacepoints)].copy()
    df['Feature'] = df['Time'] + '_' + df['Space']
    df = df.groupby(['Gene','Time','Space','Feature'], as_index=False).agg(logFC=('logFC','mean'))
    wide = df.pivot_table(index='Gene', columns='Feature', values='logFC', aggfunc='mean')
    wide = wide[[f for f in feature_order if f in wide.columns]].dropna(axis=0, how='any')
    X_raw_df = wide.copy()
    X_raw = X_raw_df.values.astype(float)
    row_mean = X_raw.mean(axis=1, keepdims=True)
    row_std = X_raw.std(axis=1, keepdims=True)
    row_std[row_std == 0] = 1.0
    X_scaled = np.nan_to_num((X_raw - row_mean) / row_std)
    X_scaled_df = pd.DataFrame(X_scaled, index=X_raw_df.index.astype(str), columns=X_raw_df.columns)
    return X_raw_df, X_scaled_df, row_mean, row_std

def vector_to_landscape(vector, columns=None):
    if columns is None:
        columns = feature_order
    s = pd.Series(np.asarray(vector, dtype=float), index=columns)
    return s.reindex(feature_order).values.reshape(len(Full_Timepoints), len(Spacepoints))

def load_category_table():
    cat = pd.read_excel(input_dir / 'putative_driver_gene_categories_12class.xlsx', sheet_name=0)
    cat['locus_ID'] = cat['locus_ID'].astype(str)
    col = 'Putative_driver_category'
    cat_map = cat.set_index('locus_ID')[col].dropna().to_dict()
    cats = sorted(pd.Series(cat_map).dropna().unique().tolist())
    palette = plt.cm.tab20(np.linspace(0, 1, max(20, len(cats))))
    color_map = {c: palette[i] for i, c in enumerate(cats)}
    default = '#2f6db3'
    return cat_map, color_map, default

def load_annotation():
    ann = pd.read_csv(input_dir / 'new_annotations_with_uniprot_names.csv')
    ann['locus_ID'] = ann['locus_ID'].astype(str)
    display_cols = ['gene_name','uniprot_gene_name','gene_name_old','KEGG_VC_number']
    def display(row):
        for c in display_cols:
            v = row.get(c, np.nan)
            if pd.notna(v) and str(v).strip() and str(v).lower() != 'nan':
                return str(v)
        return str(row['locus_ID'])
    ann['Gene_display'] = ann.apply(display, axis=1)
    text_cols = [c for c in ann.columns if c != 'locus_ID']
    ann['Annotation_text'] = ann[text_cols].astype(str).replace('nan','', regex=False).agg(' | '.join, axis=1)
    return ann

def phase_for_time(t):
    return {'1h':'Early','3h':'Early','6h':'Middle','12h':'Middle','24h':'Late'}.get(t, '')

def niche_for_space(s):
    if s == 'st': return 'stomach'
    if str(s).startswith('SI'): return 'small_intestine'
    if s == 'ce': return 'cecum'
    if s == 'co': return 'colon'
    return s

outdir = base_output_dir / 'Program_GO4'
outdir.mkdir(parents=True, exist_ok=True)
scores_path = base_output_dir / 'program_landscapes2' / 'Figure2_rank8_gene_program_scores_four_models.csv'
if not scores_path.exists():
    raise FileNotFoundError(f'Missing {scores_path}; run Figure2 notebook first.')
scores = pd.read_csv(scores_path)
cpvae = scores[scores['Model'].eq('CPVAE')].copy()
prog_cols = [c for c in cpvae.columns if c.startswith('Program')]

ann = load_annotation()
locus_to_vc = ann.set_index('locus_ID')['KEGG_VC_number'].dropna().astype(str).to_dict()
background_locus = set(ann['locus_ID'].astype(str))
go = pd.read_csv(input_dir / 'uniprot_vch_go_all.tsv', sep='\t')

go_records=[]
for _,row in go.iterrows():
    genes=str(row.get('Gene Names',''))
    vcs=re.findall(r'VC_?A?\d+', genes)
    for ontology, col in [('BP','Gene Ontology (biological process)'),('CC','Gene Ontology (cellular component)'),('MF','Gene Ontology (molecular function)')]:
        text=row.get(col, '')
        if pd.isna(text):
            continue
        for name, goid in re.findall(r'([^;\[]+)\s*\[(GO:\d{7})\]', str(text)):
            for vc in vcs:
                go_records.append({'VC':vc.replace('VCA','VC_A'),'GO_ID':goid,'GO_term':name.strip(),'Ontology':ontology})
term2gene=pd.DataFrame(go_records).drop_duplicates()
vc_to_locus={v:k for k,v in locus_to_vc.items()}
term2gene['Gene']=term2gene['VC'].map(vc_to_locus)
term2gene=term2gene.dropna(subset=['Gene']).drop_duplicates(['GO_ID','Gene'])
universe=set(term2gene['Gene']).intersection(background_locus)

def enrich(gene_list, label):
    genes=set(gene_list).intersection(universe)
    rows=[]; M=len(universe); N=len(genes)
    for (goid,term,ont), sub in term2gene.groupby(['GO_ID','GO_term','Ontology']):
        term_genes=set(sub['Gene']).intersection(universe)
        K=len(term_genes); k=len(term_genes.intersection(genes))
        if k < 2:
            continue
        p=hypergeom.sf(k-1, M, K, N)
        fold = (k / N) / (K / M) if N > 0 and K > 0 and M > 0 else np.nan
        rows.append({'Program':label,'GO_ID':goid,'GO_term':term,'Ontology':ont,'Overlap':k,'Term_size':K,'Input_size':N,'Fold_enrichment':fold,'P_value':p,'Genes':'/'.join(sorted(term_genes.intersection(genes)))})
    res=pd.DataFrame(rows)
    if res.empty:
        return res
    res=res.sort_values('P_value').reset_index(drop=True)
    m=len(res)
    res['FDR_BH']=(res['P_value']*m/(np.arange(m)+1)).clip(upper=1)
    res['minus_log10_FDR']=-np.log10(res['FDR_BH'].replace(0, np.nextafter(0,1)))
    res['log10_Fold_enrichment']=np.log10(res['Fold_enrichment'].replace(0, np.nan))
    return res

# Keep top100 and top50 GO outputs in separate folders.
for stale in outdir.glob('Figure4_CPVAE_rank8*GO_enrichment*'):
    if stale.is_file():
        stale.unlink()
for stale in outdir.glob('Figure4_CPVAE_rank8_each_program_top*_genes_for_GO.csv'):
    if stale.is_file():
        stale.unlink()

def plot_go_enrichment(top_n):
    subset_dir = outdir / f'top{top_n}'
    subset_dir.mkdir(parents=True, exist_ok=True)
    for stale in subset_dir.glob('*'):
        if stale.is_file():
            stale.unlink()

    all_results = []
    top_gene_records = []
    empty_cols = ['Program','GO_ID','GO_term','Ontology','Overlap','Term_size','Input_size','Fold_enrichment','log10_Fold_enrichment','P_value','FDR_BH','minus_log10_FDR','Genes']
    for prog in prog_cols:
        top = cpvae[['Gene','Gene_display',prog]].copy()
        top['Abs_loading'] = top[prog].abs()
        top = top.sort_values('Abs_loading', ascending=False).head(top_n)
        top['Program'] = prog
        top = top.rename(columns={prog:'Loading'})
        top_gene_records.append(top[['Program','Gene','Gene_display','Loading','Abs_loading']])
        res = enrich(top['Gene'].astype(str).tolist(), prog)
        if not res.empty:
            all_results.append(res)
            res.to_csv(subset_dir / f'Figure4_CPVAE_rank8_{prog}_top{top_n}_GO_enrichment.csv', index=False)
        else:
            pd.DataFrame(columns=empty_cols).to_csv(subset_dir / f'Figure4_CPVAE_rank8_{prog}_top{top_n}_GO_enrichment.csv', index=False)

    top_genes_df = pd.concat(top_gene_records, ignore_index=True)
    top_genes_df.to_csv(subset_dir / f'Figure4_CPVAE_rank8_each_program_top{top_n}_genes_for_GO.csv', index=False)
    combined = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame(columns=empty_cols)
    combined.to_csv(subset_dir / f'Figure4_CPVAE_rank8_top{top_n}_GO_enrichment.csv', index=False)

    plot_rows = []
    if not combined.empty:
        for prog, sub in combined.groupby('Program'):
            plot_rows.append(sub.sort_values('FDR_BH').head(8))
    plot = pd.concat(plot_rows, ignore_index=True) if plot_rows else pd.DataFrame()

    fig, axes = plt.subplots(4, 2, figsize=(17.8, 13.7), squeeze=False)
    axes_flat = axes.ravel()
    for ax, prog in zip(axes_flat, prog_cols):
        sub = plot[plot['Program'].eq(prog)].sort_values('FDR_BH', ascending=True).iloc[::-1] if not plot.empty else pd.DataFrame()
        if sub.empty:
            ax.text(0.5, 0.5, f'{prog}\nNo GO terms with overlap >= 2', ha='center', va='center', fontsize=9)
            ax.set_axis_off()
            continue
        y = np.arange(len(sub))
        sizes = 24 + 18 * sub['Overlap'].astype(float)
        color_vmin = float(sub['minus_log10_FDR'].min())
        color_vmax = float(sub['minus_log10_FDR'].max())
        if color_vmax <= color_vmin:
            color_vmax = color_vmin + 1.0
        sc = ax.scatter(
            sub['log10_Fold_enrichment'], y,
            s=sizes,
            c=sub['minus_log10_FDR'],
            cmap='viridis', vmin=color_vmin, vmax=color_vmax,
            alpha=0.88, edgecolor='black', linewidth=0.2
        )
        labels = [t[:42] + ('...' if len(t) > 42 else '') for t in sub['GO_term']]
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=5.9)
        ax.set_xlabel('log10(Fold enrichment)', fontsize=7)
        ax.set_title(f'{prog} top{top_n} genes', fontsize=9, pad=7)
        ax.tick_params(axis='x', labelsize=7, pad=2)
        ax.tick_params(axis='y', pad=2)
        ax.grid(axis='x', alpha=0.18, linewidth=0.35)

        # Align the FDR colorbar and Count legend in a shared right-side column per panel.
        cax = ax.inset_axes([1.018, 0.36, 0.026, 0.44], transform=ax.transAxes)
        cbar = fig.colorbar(sc, cax=cax)
        cbar.set_label('-log10(FDR)', fontsize=6.5, labelpad=3)
        cbar.ax.tick_params(labelsize=6, length=2)
        lax = ax.inset_axes([0.985, 0.05, 0.095, 0.22], transform=ax.transAxes)
        lax.axis('off')

        counts = sorted(sub['Overlap'].dropna().astype(int).unique().tolist())
        if len(counts) > 3:
            count_legend_values = [counts[0], counts[len(counts)//2], counts[-1]]
        else:
            count_legend_values = counts
        handles = [plt.scatter([], [], s=24 + 18*v, color='#6b7280', alpha=0.72, edgecolor='black', linewidth=0.2) for v in count_legend_values]
        lax.legend(
            handles, [str(v) for v in count_legend_values], title='Count',
            frameon=False, fontsize=5.6, title_fontsize=6.0,
            loc='center', borderaxespad=0.0,
            handletextpad=0.35, labelspacing=0.25, handlelength=0.9
        )
    for ax in axes_flat[len(prog_cols):]:
        ax.set_axis_off()
    fig.suptitle(f'GO enrichment for each CPVAE rank8 program top{top_n} contributing genes', fontsize=13, y=0.985)
    fig.subplots_adjust(top=0.92, bottom=0.09, left=0.25, right=0.95, hspace=1.06, wspace=0.95)
    save_pdf(fig, subset_dir / f'Figure4_CPVAE_rank8_top{top_n}_GO_enrichment.pdf')

for top_n in (100, 50):
    plot_go_enrichment(top_n)

print(outdir)


/Users/sidaye/Documents/python/ST_MultiCAST/Output/Program_GO4
